# v7 — controle graduado corrigido + Parkinson (Oxford)

**O que estava errado na v6.** O `ad_hoc` nasce em `[0, 2π]`, o `MinMaxScaler` comprime para
`[0, π]`, e apenas `c = 2,0` desfaz essa compressao. A v5 acertou por acidente — sua grade
terminava exatamente em 2,0. Ao estender ate 8,0, os valores geometricos viraram 1,71 e 2,86,
e o valor gerador saiu da grade: a vantagem desabou de +0,41 para −0,001.

A janela e' estreitissima. Medido: `c=1,71 &rarr; -0,001` | `1,95 &rarr; +0,355` |
`2,00 &rarr; +0,406` | `2,10 &rarr; +0,126` | `2,86 &rarr; -0,057`.

**Duas consequencias.** (i) Aqui os dados sao gerados **nativamente em `[0, π]`**, de modo que a
escala nativa e' `c = 1`, e a grade e' adensada em torno dela. (ii) A propria fragilidade e'
reportavel: o benchmark `ad_hoc`, usado como evidencia de vantagem quantica na literatura, so
e' separavel na largura de banda que reconstroi o encoding gerador. Isso vira uma secao do
artigo, nao uma nota de rodape.

**Parkinson (Oxford).** 195 gravacoes, 22 features, 32 sujeitos com 6-7 gravacoes cada. Tem
estrutura de sujeito como a SVD — mesmo protocolo. Limitacao dura: **apenas 8 controles
saudaveis** contra 24 com Parkinson, o que restringe o numero de folds e alarga os IC.

In [ ]:
# !pip install pennylane -q
import numpy as np, pandas as pd, time, re
import pennylane as qml
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize_scalar
from scipy.stats import spearmanr, t as tdist
from IPython.display import HTML, display
import warnings; warnings.filterwarnings("ignore")

BG, FG, OR = "#0d0d0d", "#e8e8e8", "#ff6b00"
FONT = "'JetBrains Mono','Fira Code',monospace"

# Painel escuro com acento laranja no lugar de print cru.
def painel(titulo, linhas, tabela=None, destaque=None):
    corpo = "".join(f"<div style='margin:.28em 0'><span style='color:{OR}'>$</span> {l}</div>"
                    for l in linhas)
    tab = ""
    if tabela is not None:
        tab = (tabela.style.hide(axis="index").set_table_styles([
            {"selector": "th", "props": [("color", OR), ("text-align", "right"),
                                         ("border-bottom", f"1px solid {OR}"), ("padding", "4px 12px")]},
            {"selector": "td", "props": [("text-align", "right"), ("padding", "3px 12px"),
                                         ("border-bottom", "1px solid #222")]}])
            .set_table_attributes(f"style='border-collapse:collapse;font-family:{FONT};"
                                  f"font-size:12px;color:{FG};margin:.8em 0 .2em 0'").to_html())
    dest = (f"<div style='margin-top:.9em;padding:.6em .8em;border-left:3px solid {OR};"
            f"background:#1a1206;color:{FG}'>{destaque}</div>") if destaque else ""
    display(HTML(f"<div style='background:{BG};color:{FG};font-family:{FONT};font-size:13px;"
                 f"padding:1.1em 1.3em;border:1px solid #222;border-radius:6px'>"
                 f"<div style='color:{OR};font-size:11px;letter-spacing:.16em;margin-bottom:.7em'>"
                 f"&#9632; {titulo.upper()}</div>{corpo}{tab}{dest}</div>"))

def _sf(q, reps, ent):
    dev = qml.device("default.qubit", wires=q)
    @qml.qnode(dev)
    def circ(x):
        for _ in range(reps):
            for i in range(q):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            if ent:
                for i in range(q-1):
                    qml.CNOT(wires=[i, i+1])
                    qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                    qml.CNOT(wires=[i, i+1])
        return qml.state()
    return circ

def states(X, q, reps=2, ent=True, batch=512):
    f = _sf(q, reps, ent)
    return np.vstack([np.asarray(f(X[s:s+batch])) for s in range(0, len(X), batch)])

def gram_fid(A, B): return np.abs(A @ B.conj().T)**2

def fit_surrogate(K, A):
    D2 = ((A[:, None, :]-A[None, :, :])**2).sum(-1)
    r = minimize_scalar(lambda lg: np.linalg.norm(K-np.exp(-np.exp(lg)*D2)),
                        bounds=(-8, 8), method="bounded")
    return float(np.exp(r.x)), float(r.fun)/np.sqrt(K.size)

## 1. Grade de largura de banda

Grossa em escala geometrica **mais** um trecho denso em torno de `c = 1`, que e' a escala nativa
do `ad_hoc` gerado nesta versao. Sem o trecho denso, o controle positivo falha por resolucao de
grade — foi exatamente o que aconteceu na v6.

In [ ]:
SC_GROSSA = np.geomspace(0.01, 4.0, 12)
SC_DENSA = np.linspace(0.80, 1.25, 10)          # em torno da escala nativa
SCALES = np.round(np.unique(np.concatenate([SC_GROSSA, SC_DENSA, [1.0]])), 4)
painel("grade de bandwidth",
       [f"{len(SCALES)} valores &middot; grossa de {SC_GROSSA.min():.2f} a {SC_GROSSA.max():.0f}"
        f" + densa de {SC_DENSA.min():.2f} a {SC_DENSA.max():.2f}",
        f"c=1,0 presente: <b>{1.0 in SCALES}</b>"],
       destaque="A v6 falhou porque o valor gerador (c=2,0 naquela parametrizacao) nao estava "
                "na grade. Verifique sempre que a escala nativa esta contida antes de "
                "interpretar um resultado nulo do controle.")

## 2. Estimador

Vantagem = AUC quantico &minus; AUC do surrogate, com a bandwidth escolhida por
**leave-one-fold-out**: o `c` usado no fold `i` vem da media dos demais. `q` fica fixo — a v6
selecionava `(q, c)` conjuntamente, o que adiciona ruido sem necessidade.

In [ ]:
def vantagem(X, y, grupos, q, folds=10, npc=None, seed=7, reps=2):
    rows = []
    sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    for f, (tr, te) in enumerate(sg.split(X, y, grupos)):
        imp = SimpleImputer(strategy="median").fit(X[tr])
        Xtr, Xte = imp.transform(X[tr]), imp.transform(X[te])
        ytr, yte = y[tr], y[te]
        k = min(q, Xtr.shape[1])
        sel = SelectKBest(f_classif, k=k).fit(Xtr, ytr)
        sc = MinMaxScaler((0, np.pi)).fit(sel.transform(Xtr))
        A0 = sc.transform(sel.transform(Xtr))
        B0 = np.clip(sc.transform(sel.transform(Xte)), 0, np.pi)
        for cv in SCALES:
            A, B = A0*cv, B0*cv
            S1, S2 = states(A, k, reps), states(B, k, reps)
            K = gram_fid(S1, S1)
            m = SVC(kernel="precomputed").fit(K, ytr)
            gs, resid = fit_surrogate(K, A)
            ms = SVC(kernel="rbf", gamma=gs).fit(A, ytr)
            rows.append(dict(fold=f, c=cv, resid=resid,
                             aq=roc_auc_score(yte, m.decision_function(gram_fid(S2, S1))),
                             asr=roc_auc_score(yte, ms.decision_function(B))))
    d = pd.DataFrame(rows)
    v, cs = [], []
    for f in sorted(d.fold.unique()):
        c_star = d[d.fold != f].groupby("c").aq.mean().idxmax()
        r = d[(d.fold == f) & (d.c == c_star)].iloc[0]
        v.append(r.aq - r.asr); cs.append(c_star)
    v = np.array(v); n = len(v)
    se = v.std(ddof=1)/np.sqrt(n); tc = tdist.ppf(0.975, n-1)
    return d, dict(vantagem=v.mean(), lo=v.mean()-tc*se, hi=v.mean()+tc*se,
                   c_mediano=float(np.median(cs)), folds=n)

## 3. Controle positivo graduado — escala nativa

Dados gerados em `[0, π]`, portanto `c = 1` reproduz o encoding gerador. O `gap` controla a
margem: alto = fortemente separavel, zero = sem estrutura (deve dar vantagem nula).

In [ ]:
def gera_adhoc(n=500, dim=4, gap=0.3, seed=0, hi=np.pi):
    r = np.random.default_rng(seed)
    dev = qml.device("default.qubit", wires=dim)
    @qml.qnode(dev)
    def phi(x):
        for _ in range(2):
            for i in range(dim):
                qml.Hadamard(wires=i); qml.RZ(2.0*x[..., i], wires=i)
            for i in range(dim-1):
                qml.CNOT(wires=[i, i+1])
                qml.RZ(2.0*(np.pi-x[..., i])*(np.pi-x[..., i+1]), wires=i+1)
                qml.CNOT(wires=[i, i+1])
        return qml.state()
    V = np.linalg.qr(r.normal(size=(2**dim,)*2) + 1j*r.normal(size=(2**dim,)*2))[0]
    D = np.diag([1.]*(2**(dim-1)) + [-1.]*(2**(dim-1)))
    O = V.conj().T @ D @ V
    X, y = [], []
    while len(X) < n:
        cand = r.uniform(0, hi, (4000, dim))
        S = np.asarray(phi(cand))
        val = np.einsum('ni,ij,nj->n', S.conj(), O, S).real
        for xx, vv in zip(cand[np.abs(val) > gap], val[np.abs(val) > gap]):
            X.append(xx); y.append(int(vv > 0))
            if len(X) >= n: break
    return np.array(X), np.array(y)

GAPS = [0.30, 0.20, 0.12, 0.07, 0.03, 0.0]
lin, t0 = [], time.time()
for gp in GAPS:
    Xg, yg = gera_adhoc(n=500, dim=4, gap=gp)
    _, r = vantagem(Xg, yg, np.arange(len(yg)), q=4, folds=10)
    lin.append(dict(gap=gp, prev=round(yg.mean(), 3), vantagem=round(r["vantagem"], 4),
                    ic95_lo=round(r["lo"], 4), ic95_hi=round(r["hi"], 4),
                    c_escolhido=r["c_mediano"], detectada=bool(r["lo"] > 0)))
    print(f"gap={gp} ok — {time.time()-t0:.0f}s")
grad = pd.DataFrame(lin)

det = grad[grad.detectada & (grad.gap > 0)]
menor = det.vantagem.min() if len(det) else None
nulo = grad[grad.gap == 0.0]
aviso = ""
if len(nulo) and bool(nulo.detectada.iloc[0]):
    aviso = ("<br><b>ATENCAO</b>: vantagem detectada em gap=0, onde nao ha estrutura. "
             "Ha vies no procedimento de selecao — nao use nenhum numero deste notebook.")
if len(det) and abs(det.c_escolhido - 1.0).max() > 0.3:
    aviso += ("<br><b>ATENCAO</b>: o c escolhido se afastou da escala nativa; confira se a "
              "grade densa esta cobrindo o otimo.")
painel("controle positivo graduado (escala nativa)",
       [f"dim=4, n=500, {len(GAPS)} niveis, 10 folds, escala nativa c=1",
        f"menor vantagem com IC acima de zero: <b>{menor if menor is not None else 'nenhuma'}</b>"],
       tabela=grad,
       destaque=(f"Reporte: <i>o protocolo detecta vantagens a partir de "
                 f"{menor if menor is not None else '[?]'} de AUC com 10 folds e n=500</i>. "
                 "Abaixo disso, ausencia de vantagem nos corpora reais nao e' conclusiva."
                 + aviso))

## 4. Fragilidade do benchmark ad hoc

Varredura fina em torno da escala nativa, sem selecao: quao rapido a separabilidade some quando
a bandwidth se afasta do valor gerador. Esta figura sozinha e' um argumento sobre o uso do
`ad_hoc` como evidencia de vantagem quantica na literatura.

In [ ]:
Xf, yf = gera_adhoc(n=500, dim=4, gap=0.30)
d_f, _ = vantagem(Xf, yf, np.arange(len(yf)), q=4, folds=5)
curva = d_f.groupby("c")[["aq", "asr"]].mean()
curva["vantagem"] = (curva.aq - curva.asr).round(4)
curva = curva.round(4).reset_index()

pico = curva.loc[curva.vantagem.idxmax()]
meia = curva[curva.vantagem >= pico.vantagem/2].c
painel("largura da janela de separabilidade",
       [f"pico em c = <b>{pico.c}</b> com vantagem {pico.vantagem:+.4f}",
        f"faixa com pelo menos metade do pico: c &isin; [{meia.min()}, {meia.max()}]"],
       tabela=curva,
       destaque="Se a janela for estreita, o <i>ad_hoc</i> nao mede vantagem quantica em geral: "
                "mede se o experimentador acertou a bandwidth geradora. E' um resultado sobre o "
                "benchmark, e vale uma secao do artigo.")

## 5. Parkinson (Oxford)

Agrupamento por sujeito extraido do campo `name` (`phon_R01_S01_1` &rarr; `phon_R01_S01`).
Ignorar isso vazaria o mesmo sujeito entre treino e teste.

**Limitacao dura:** 8 sujeitos controle contra 24 com Parkinson. Com folds por sujeito, sobram
1 ou 2 controles por fold de teste. Os IC serao largos — isso e' propriedade do corpus, nao do
protocolo, e vai para as limitacoes do artigo.

In [ ]:
def carrega_parkinsons(path="parkinsons.data"):
    d = pd.read_csv(path)
    subj = d.name.str.extract(r'(phon_R\d+_S\d+)')[0]
    F = [c for c in d.columns if c not in ("name", "status")]
    return d[F].values.astype(float), d.status.values.astype(int), subj.values, F

Xp, yp, gp_, Fp = carrega_parkinsons()
n_sub = pd.Series(gp_).nunique()
por_classe = pd.DataFrame({"s": gp_, "y": yp}).groupby("s").y.first().value_counts().to_dict()

FOLDS_PARK = 4
lin = []
for q in [4, 6, 8, 10]:
    _, r = vantagem(Xp, yp, gp_, q=q, folds=FOLDS_PARK)
    lin.append(dict(qubits=q, vantagem=round(r["vantagem"], 4), ic95_lo=round(r["lo"], 4),
                    ic95_hi=round(r["hi"], 4), c_mediano=r["c_mediano"],
                    detectada=bool(r["lo"] > 0)))
park = pd.DataFrame(lin)

d_full, _ = vantagem(Xp, yp, gp_, q=8, folds=FOLDS_PARK)
a = d_full.groupby("c")[["aq", "asr"]].mean()
rho, pv = spearmanr(a.aq, a.asr)
teto = a.aq.max() > 0.97
painel("Parkinson (Oxford)",
       [f"{Xp.shape[0]} gravacoes &middot; {len(Fp)} features &middot; {n_sub} sujeitos",
        f"sujeitos por classe: {por_classe} &nbsp;|&nbsp; {FOLDS_PARK} folds",
        f"AUC maximo (q=8) = {a.aq.max():.4f} &nbsp;|&nbsp; "
        f"Spearman(auc_q, auc_surr) = <b>{rho:+.3f}</b> (p={pv:.2g})"],
       tabela=park,
       destaque=("<b>Saturado</b> — como o WDBC, nao serve como evidencia de ausencia de "
                 "vantagem." if teto else
                 "Nao saturado: o corpus discrimina metodos e pode entrar na sintese. "
                 "Reporte os IC largos como limitacao de tamanho amostral."))

## 6. Leitura

1. Se o controle graduado der um limiar de deteccao **acima** das vantagens tipicamente
   reivindicadas na literatura (0,02&ndash;0,05), o artigo nao pode afirmar ausencia de efeito —
   tem que afirmar que o efeito, se existe, esta abaixo do que este desenho enxerga. E' uma
   conclusao mais fraca, e a unica honesta.
2. A fragilidade do `ad_hoc` (secao 4) e' contribuicao propria, independente do resto.
3. Corpora reais uteis ate aqui: SVD e — se nao saturar — Parkinson. WDBC saiu.